<a href="https://colab.research.google.com/github/NBall65097/Fantasy-Story-Weaver/blob/main/Fantasy_Story_Weaver.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers==0.0.28.post2 trl peft accelerate bitsandbytes

In [ ]:
from huggingface_hub import login
from google.colab import userdata
token = userdata.get('HuggingFaceWrite')
login(token) # Connect to HuggingFace account

In [ ]:
import datasets # datasets function for importing data

In [ ]:
# # This was used to debug issues with the AI-generated training data formatting.

# import json

# file_path = '/content/data/FSTData.jsonl'

# with open(file_path, 'r', encoding='utf-8') as f:
#     for i, line in enumerate(f, 1):
#         line = line.strip()
#         if not line:
#             continue
#         try:
#             json.loads(line)
#         except json.JSONDecodeError as e:
#             print(f"First error at row {i}: {e}")
#             print(f"Problem line (first 200 chars): {line[:200]}...")
#             print(f"Full line length: {len(line)}")
#             break
#     else:
#         print("All lines appear valid!")

In [ ]:
#filepath = "/content/data/FSTData.jsonl"
#data = datasets.load_dataset("json",data_files=filepath,split='train')
#data.push_to_hub('NBall65097/fantasy-storyweaver-data') # Save dataset to HuggingFace

In [ ]:
import torch
from datasets import load_dataset
import unsloth
from trl import SFTTrainer, SFTConfig
from unsloth import FastLanguageModel

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    "microsoft/Phi-3.5-mini-instruct",
    dtype=None,  # auto
    load_in_4bit=True,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

In [ ]:
import json
dataset = load_dataset('NBall65097/fantasy-storyweaver-data',split="train")

In [ ]:
from unsloth.chat_templates import apply_chat_template

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="phi-3")

def format_phi35_data(examples):
  formatted_texts = []
  for msg_list in examples["messages"]:
    text = tokenizer.apply_chat_template(
        msg_list,
        tokenize=False,
        add_generation_prompt=False,
    )
    formatted_texts.append(text)
  return {"text":formatted_texts}

In [ ]:
dataset = dataset.map(
    format_phi35_data,
    batched=True,
    batch_size=1000,          # adjust based on RAM
    num_proc=2,               # or 4 if you have enough CPU cores
    desc="Formatting Phi-3.5 chat examples",
)

In [ ]:
data = dataset.train_test_split(test_size=0.1, train_size=0.9)
traindata, testdata = data["train"], data["test"]

In [ ]:
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=traindata,
    eval_dataset=testdata,
    args=SFTConfig(
        output_dir="/content/output",
        num_train_epochs=2,
        learning_rate=2e-4,
        max_seq_length=2048,
        dataset_text_field="text",  # or use formatting_func for chat template
        packing=True,  # packs multiple examples → huge speed win
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        warmup_ratio=0.03,
        fp16=True,
        gradient_checkpointing="unsloth",
        optim="adamw_8bit",
        logging_steps=10,
        save_strategy="epoch",
        #save_steps=200,
        report_to="none",
    )  # TrainingArguments: 1–3 epochs, lr=2e-4, batch=4–8 (gradient accum=4), warmup, etc.
)

In [ ]:
trainer.train()

In [ ]:
model.save_pretrained("fantasy-story-weaver-lora")          # saves adapter only (~50–200 MB)
tokenizer.save_pretrained("fantasy-story-weaver-lora")

In [ ]:
model.save_pretrained_merged(
    "fantasy-story-weaver-merged",
    tokenizer,
    save_method = "merged_16bit",
)

In [ ]:
# # This just pushes the LoRA adapter. Small, fast, but not the full model.

# model.push_to_hub("NBall65097/fantasy-story-weaver-lora")
# tokenizer.push_to_hub("NBall65097/fantasy-story-weaver-lora")

In [ ]:
# # This pushes the full, merged model.

# model.push_to_hub_merged(
#     "NBall65097/fantasy-story-weaver",
#     tokenizer,
#     save_method = "merged_16bit",
#     token = True
# )

**Evaluation**

In [ ]:
eval_results = trainer.evaluate()
print(eval_results)